# Efficient ResNet Challenge — Big-teacher distillation (≤ 5M params submitted)

**Runs inside a Kaggle Notebook** (competition data is attached automatically, nothing to download).
Before running: right sidebar → **Session options → Accelerator → GPU T4 x2** (or P100).

Pipeline:
1. Find the competition files under `/kaggle/input`
2. Load the 50k training images + labels from the competition files, hold out a validation split
3. Wide pre-activation ResNet-18 variant, **~4.8M params** (asserted below)
4. Fast training: all data on GPU, GPU-side augmentation (crop + flip + cutout), mixed precision, OneCycle LR, label smoothing
5. **CutMix** (mixes patches of two training images + their labels), **EMA** (running average of the weights, used for prediction), **SAM** optimizer (seeks flat minima that generalize better)
6. **Big-teacher distillation**: first trains ONE large teacher (~19M params, 2× wider, from scratch on the same 50k images, strong augmentation), then trains the ≤5M student on the true labels *and* the teacher's soft predictions. Only the student is submitted; the 5M limit applies to it. Teachers trained from scratch on the permitted data are explicitly allowed.
7. **TrivialAugment** (random colour / geometry ops) on top of crop, flip, Cutout and CutMix, for both models
8. Predict test set with multi-view TTA (flips + small shifts) → `submission.csv`

**How to run:** Accelerator → GPU T4 x2, then **Save Version → Save & Run All (Commit)**. Roughly 5–7 hours; you can close the tab. The log prints seconds per epoch, so the total can be estimated after a few minutes.

**Safety net:** the teacher is saved as `big_teacher.pt` as soon as it finishes. If the run dies later, attach this notebook's output as an input (Add Input → Your Work) and run again: the saved teacher is found and reused, skipping its training.

In [ ]:
!nvidia-smi -L
import os, glob, time, math, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import torchvision

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)
assert device == 'cuda', 'Turn on the GPU: Session options > Accelerator > GPU'

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

## 1. Find the competition files

In [ ]:
DATA_DIR = Path('/kaggle/input')
dirs = {}
for p in DATA_DIR.rglob('*.png'):
    dirs.setdefault(p.parent, []).append(p)
for d, ps in sorted(dirs.items(), key=lambda t: -len(t[1]))[:15]:
    print(f'{len(ps):6d} png  {d}')
csv_files = list(DATA_DIR.rglob('*.csv'))
for c in csv_files:
    print('csv:', c, '| columns:', list(pd.read_csv(c, nrows=2).columns))

## 2. Load data

Loads the training images and labels from the competition files (handles a labels CSV *or* one folder per class),
and the test images in `sample_submission` order. Takes ~1–2 minutes.

In [ ]:
import re
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
def load_img(p):
    return torch.from_numpy(np.array(Image.open(p).convert('RGB'))).permute(2, 0, 1)
def num(s):
    d = re.findall(r'\d+', str(s)); return int(d[-1]) if d else None

sample = pd.read_csv(next(c for c in csv_files if 'sample' in c.name.lower()))

# ---- test images: the folder with exactly as many PNGs as sample_submission rows ----
TEST_DIR = None   # set manually if auto-detect fails
if TEST_DIR is None:
    cand = [d for d, ps in dirs.items() if len(ps) == len(sample)] or [d for d in dirs if 'test' in str(d).lower()]
    TEST_DIR = cand[0]
tpaths = {num(p.stem): p for p in dirs[Path(TEST_DIR)]}
missing = [i for i in sample['id'] if num(i) not in tpaths]
assert not missing, f'{len(missing)} test ids have no image, e.g. {missing[:5]}'
X_test = torch.stack([load_img(tpaths[num(i)]) for i in sample['id']])
print('TEST_DIR =', TEST_DIR, '| test:', tuple(X_test.shape))

# ---- training images ----
train_pngs = [p for d, ps in dirs.items() if d != Path(TEST_DIR) for p in ps]
label_csv = None
for c in csv_files:
    df = pd.read_csv(c)
    if 'sample' not in c.name.lower() and 'label' in df.columns and len(df) >= 10000:
        label_csv = df; print('labels from', c); break

if label_csv is not None:
    key_col = [k for k in label_csv.columns if k != 'label'][0]
    by_stem = {p.stem: p for p in train_pngs}
    by_num = {num(p.stem): p for p in train_pngs}
    def find(k):
        k = str(k); stem = Path(k).stem
        return by_stem.get(stem) or by_num.get(num(stem))
    pairs = [(find(k), lab) for k, lab in zip(label_csv[key_col], label_csv['label'])]
    pairs = [(p, lab) for p, lab in pairs if p is not None]
    labels = [lab if str(lab).isdigit() else CLASSES.index(str(lab).lower()) for _, lab in pairs]
    train_list = [p for p, _ in pairs]
else:
    # one folder per class (folder named by class name or 0-9)
    def cls(p):
        n = p.parent.name.lower()
        return int(n) if n.isdigit() else (CLASSES.index(n) if n in CLASSES else None)
    train_list = [p for p in train_pngs if cls(p) is not None]
    labels = [cls(p) for p in train_list]

assert len(train_list) > 10000, 'Could not find labelled training images -- paste the output of the cell above to get help'
X_all = torch.stack([load_img(p) for p in train_list])
y_all = torch.tensor([int(l) for l in labels])
print('train:', tuple(X_all.shape), '| per class:', y_all.bincount().tolist())

In [ ]:
# Sanity check: look at a few training and test images
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for k in range(8):
    ax[0, k].imshow(X_all[k].permute(1, 2, 0)); ax[0, k].set_title(CLASSES[y_all[k]], fontsize=8)
    ax[1, k].imshow(X_test[k].permute(1, 2, 0)); ax[1, k].set_title(f'test id {sample.id[k]}', fontsize=8)
for a in ax.flat: a.axis('off')
plt.show()

## 3. Train/val split, GPU augmentation

In [ ]:
VAL_SIZE = 0             # 0 = final run on all 50k images; 5000 = experiment mode (shows val accuracy)

perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
val_idx, tr_idx = perm[:VAL_SIZE], perm[VAL_SIZE:]

MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1)
def to_gpu(x):  # uint8 -> normalized float on GPU
    return ((x.to(device).float() / 255) - MEAN) / STD

Xtr, ytr = to_gpu(X_all[tr_idx]), y_all[tr_idx].to(device)
Xtr_u8 = X_all[tr_idx].to(device)            # raw pixels, for TrivialAugment (needs uint8)
Xva, yva = to_gpu(X_all[val_idx]), y_all[val_idx].to(device)
Xte = to_gpu(X_test)
print('train', Xtr.shape, '| val', Xva.shape, '| test', Xte.shape)

from torchvision.transforms import v2
_trivial = v2.TrivialAugmentWide()
def trivial_aug(x_u8, chunk=16):
    # TrivialAugment: one random op (colour, contrast, rotate, shear, ...) with random strength per chunk of 16 images
    return torch.cat([_trivial(c) for c in x_u8.split(chunk)])

def normalize(x_u8):
    return ((x_u8.float() / 255) - MEAN) / STD

def augment(x, pad=4, cutout=16):
    # Random crop (reflect pad), horizontal flip, cutout — all on GPU, per image.
    N, C, H, W = x.shape
    xp = F.pad(x, (pad,) * 4, mode='reflect')
    i = torch.randint(0, 2 * pad + 1, (N,), device=x.device)
    j = torch.randint(0, 2 * pad + 1, (N,), device=x.device)
    ar = torch.arange(H, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None].expand(N, C, H, W)
    cols = (j[:, None] + ar)[:, None, None, :].expand(N, C, H, W)
    x = xp[torch.arange(N, device=x.device)[:, None, None, None],
           torch.arange(C, device=x.device)[None, :, None, None], rows, cols]
    flip = torch.rand(N, device=x.device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    if cutout:
        cy = torch.randint(0, H, (N,), device=x.device)[:, None, None]
        cx = torch.randint(0, W, (N,), device=x.device)[:, None, None]
        yy, xx = ar[None, :, None], ar[None, None, :]
        m = ((yy - cy).abs() < cutout // 2) & ((xx - cx).abs() < cutout // 2)
        x = x * (~m)[:, None].float()
    return x

## 4. Model: wide pre-activation ResNet (≤ 5M params)

`F(x)` = BN → ReLU → Conv3×3 → BN → ReLU → Conv3×3; `S(x)` = identity, or a 1×1 conv projection when shape changes.
Knobs to experiment with: `widths`, `blocks`, kernel size, dropout before the classifier.

In [ ]:
class PreActBlock(nn.Module):
    def __init__(self, cin, cout, stride=1, k=3):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(cin)
        self.conv1 = nn.Conv2d(cin, cout, k, stride, k // 2, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, k, 1, k // 2, bias=False)
        self.shortcut = None
        if stride != 1 or cin != cout:
            self.shortcut = nn.Conv2d(cin, cout, 1, stride, bias=False)   # learned projection S(x)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        s = self.shortcut(out) if self.shortcut is not None else x       # identity S(x)
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + s                                                   # F(x) + S(x)

class ResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256, 352), blocks=(2, 2, 2, 1), num_classes=10, drop=0.0):
        super().__init__()
        self.stem = nn.Conv2d(3, widths[0], 3, 1, 1, bias=False)
        layers, cin = [], widths[0]
        for s, (w, n) in enumerate(zip(widths, blocks)):
            for b in range(n):
                layers.append(PreActBlock(cin, w, stride=2 if (b == 0 and s > 0) else 1))
                cin = w
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(cin)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(cin, num_classes)

    def forward(self, x):
        x = self.layers(self.stem(x))
        x = F.relu(self.bn(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(self.drop(x))

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

model = ResNet().to(device).to(memory_format=torch.channels_last)
n_params = count_params(model)
print(f'Trainable parameters: {n_params:,}')
assert n_params <= 5_000_000, 'Over the 5M budget!'

## 5. Training setup

- **CutMix** (`USE_CUTMIX`): with probability `CUTMIX_PROB`, a random box from another image in the batch is pasted in and the label is mixed by box area. Off for the last `CUTMIX_OFF_LAST` epochs.
- **EMA** (`USE_EMA`): exponential moving average of the weights (`EMA_DECAY`), used for evaluation and predictions.
- **SAM** (`USE_SAM`): Sharpness-Aware Minimization, radius `SAM_RHO`; favours flat minima. ~2× time per epoch.
- **Distillation** (student only): loss = `(1 - KD_ALPHA) * normal loss + KD_ALPHA * T² * KL(student || teacher ensemble)` at temperature `KD_T`. Teachers see exactly the same augmented image as the student.
- **TTA** (`TTA_FULL`): final predictions average 10 views (original + 4 two-pixel shifts, each also flipped).

In [ ]:
import copy

TEACHER_WIDTHS  = (128, 256, 512, 704)   # ~19.2M params - the teacher has no size limit
TEACHER_EPOCHS  = 200
TEACHER_SAM     = False   # off for the teacher to save time (it is 4x bigger)
EPOCHS          = 250     # student
BATCH           = 256
LR              = 0.2
WD              = 5e-4
SMOOTHING       = 0.1

USE_TRIVIALAUG  = True
USE_CUTMIX      = True
CUTMIX_PROB     = 0.5
CUTMIX_OFF_LAST = 10
USE_EMA         = True
EMA_DECAY       = 0.999
USE_SAM         = True    # student
SAM_RHO         = 0.05
TTA_FULL        = True
KD_ALPHA        = 0.8
KD_T            = 4.0

criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)

def cutmix(x, y):
    N, _, H, W = x.shape
    perm = torch.randperm(N, device=x.device)
    lam = float(np.random.beta(1.0, 1.0))
    ch, cw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2 = max(cy - ch // 2, 0), min(cy + ch // 2, H)
    x1, x2 = max(cx - cw // 2, 0), min(cx + cw // 2, W)
    x = x.clone()
    x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return x, y, y[perm], lam

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point: v.lerp_(msd[k], 1 - self.decay)
            else: v.copy_(msd[k])

@torch.no_grad()
def predict_logits(m, X, bs=1000, tta='flip'):
    # tta: None = single view, 'flip' = original + flip, 'full' = original + 4 shifts (2 px), each with its flip
    m.eval(); out = []
    shifts = [(0, 0)] + ([(-2, 0), (2, 0), (0, -2), (0, 2)] if tta == 'full' else [])
    for k in range(0, len(X), bs):
        xp = F.pad(X[k:k + bs], (2, 2, 2, 2), mode='reflect')
        lo = 0
        for dy, dx in shifts:
            v = xp[:, :, 2 + dy:34 + dy, 2 + dx:34 + dx].contiguous(memory_format=torch.channels_last)
            with torch.autocast('cuda', dtype=torch.float16):
                lo = lo + m(v).float()
                if tta: lo = lo + m(v.flip(3)).float()
        out.append(lo)
    return torch.cat(out)

@torch.no_grad()
def teacher_logits(teachers, x):
    with torch.autocast('cuda', dtype=torch.float16):
        return torch.stack([t(x).float() for t in teachers])      # (n_teachers, N, 10)

def sam_step(model, opt, scaler, xb, loss_fn):
    # SAM with mixed precision: grads at w -> perturb to w + rho*g/|g| -> grads there -> restore w -> step.
    with torch.autocast('cuda', dtype=torch.float16):
        logits = model(xb); loss = loss_fn(logits)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    inv_scale = 1.0 / scaler.get_scale()
    params = [p for p in model.parameters() if p.grad is not None]
    gnorm = torch.norm(torch.stack([p.grad.norm(2) for p in params]), 2) * inv_scale
    eps = None
    if torch.isfinite(gnorm):
        with torch.no_grad():
            eps = [p.grad * (inv_scale * SAM_RHO / (gnorm + 1e-12)) for p in params]
            for p, e in zip(params, eps): p.add_(e)
    with torch.autocast('cuda', dtype=torch.float16):
        loss2 = loss_fn(model(xb))
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss2).backward()
    if eps is not None:
        with torch.no_grad():
            for p, e in zip(params, eps): p.sub_(e)
    scaler.step(opt); scaler.update()
    return logits, loss

def train_model(seed, epochs, teachers=(), name='model', widths=(64, 128, 256, 352), use_sam=True):
    # Trains one ResNet from scratch. With teachers, adds the distillation loss. Returns (final model, history).
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = ResNet(widths=widths).to(device).to(memory_format=torch.channels_last)
    print(f'{name}: {sum(p.numel() for p in model.parameters()):,} parameters')
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, nesterov=True, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=math.ceil(len(Xtr) / BATCH),
                                                pct_start=0.15, div_factor=10, final_div_factor=200)
    scaler = torch.amp.GradScaler('cuda')
    ema = EMA(model, EMA_DECAY) if USE_EMA else None
    history = []
    print(f'===== training {name} (seed {seed}, {epochs} epochs, {len(teachers)} teachers, SAM={use_sam}) =====')
    for epoch in range(epochs):
        model.train(); t0 = time.time()
        use_cm = USE_CUTMIX and epoch < epochs - CUTMIX_OFF_LAST
        order = torch.randperm(len(Xtr), device=device)
        tot_loss, correct = 0.0, 0
        for k in range(0, len(Xtr), BATCH):
            idx = order[k:k + BATCH]
            xb = normalize(trivial_aug(Xtr_u8[idx])) if USE_TRIVIALAUG else Xtr[idx]
            xb, yb = augment(xb), ytr[idx]
            mixed = use_cm and np.random.rand() < CUTMIX_PROB
            if mixed:
                xb, ya, yb2, lam = cutmix(xb, yb)
            xb = xb.contiguous(memory_format=torch.channels_last)
            if mixed:
                base_fn = lambda lo: lam * criterion(lo, ya) + (1 - lam) * criterion(lo, yb2)
            else:
                base_fn = lambda lo: criterion(lo, yb)
            if teachers:
                t_prob = (teacher_logits(teachers, xb) / KD_T).softmax(2).mean(0)
                loss_fn = lambda lo: (1 - KD_ALPHA) * base_fn(lo) + KD_ALPHA * KD_T ** 2 * \
                    F.kl_div(F.log_softmax(lo.float() / KD_T, 1), t_prob, reduction='batchmean')
            else:
                loss_fn = base_fn
            if use_sam:
                logits, loss = sam_step(model, opt, scaler, xb, loss_fn)
            else:
                with torch.autocast('cuda', dtype=torch.float16):
                    logits = model(xb); loss = loss_fn(logits)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
            sched.step()
            if ema: ema.update(model)
            tot_loss += loss.item() * len(idx)
            correct += (logits.argmax(1) == yb).sum().item()
        row = dict(epoch=epoch + 1, loss=tot_loss / len(Xtr), train_acc=correct / len(Xtr))
        if VAL_SIZE and ((epoch + 1) % 10 == 0 or epoch + 1 == epochs):
            row['val_acc'] = (predict_logits(model, Xva).argmax(1) == yva).float().mean().item()
            if ema: row['val_acc_ema'] = (predict_logits(ema.model, Xva).argmax(1) == yva).float().mean().item()
        history.append(row)
        if (epoch + 1) % 5 == 0 or epoch == 0 or 'val_acc' in row:
            print(f"[{name}] ep {row['epoch']:3d} | loss {row['loss']:.3f} | train {row['train_acc']:.4f}"
                  + (f" | val {row['val_acc']:.4f}" if 'val_acc' in row else '')
                  + (f" | val_ema {row['val_acc_ema']:.4f}" if 'val_acc_ema' in row else '')
                  + (' | cutmix' if use_cm else '') + f" | {time.time() - t0:.1f}s/epoch")
    final = ema.model if ema else model
    final.eval()
    for p in final.parameters(): p.requires_grad_(False)
    torch.save(final.state_dict(), f'/kaggle/working/{name}.pt')
    return final, history

## 6. Train the big teacher, then the student

In [ ]:
t_start = time.time()
saved = sorted(Path('/kaggle/input').rglob('big_teacher.pt'))
if saved:
    teacher = ResNet(widths=TEACHER_WIDTHS).to(device).to(memory_format=torch.channels_last)
    teacher.load_state_dict(torch.load(saved[0], map_location=device)); teacher.eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    print('Reusing saved teacher:', saved[0])
else:
    teacher, _ = train_model(seed=100, epochs=TEACHER_EPOCHS, name='big_teacher', widths=TEACHER_WIDTHS, use_sam=TEACHER_SAM)
    print(f'--- teacher done, {(time.time() - t_start) / 60:.0f} min elapsed')
teachers = [teacher]

chk = torch.arange(2000)
acc = (predict_logits(teacher, to_gpu(X_all[chk])).argmax(1).cpu() == y_all[chk]).float().mean()
print(f'Teacher sanity check on 2,000 training images: {acc:.4f} (should be ~0.97-1.00)')

final_model, history = train_model(seed=0, epochs=EPOCHS, teachers=teachers, name='student', use_sam=USE_SAM)
n_final = sum(p.numel() for p in final_model.parameters())
print(f'All training done in {(time.time() - t_start) / 60:.0f} min. Parameters of the submitted student: {n_final:,} (limit 5,000,000)')
assert n_final <= 5_000_000

In [ ]:
# Validation summary (only when VAL_SIZE > 0) - useful for the report: teachers alone vs teacher ensemble vs distilled student
if VAL_SIZE >= 100:
    from sklearn.metrics import confusion_matrix
    for k, t in enumerate(teachers):
        print(f'teacher (big)    val (full TTA): {(predict_logits(t, Xva, tta="full").argmax(1) == yva).float().mean():.4f}')
    for mode in [None, 'flip', 'full']:
        print(f'student          val (TTA={mode}): {(predict_logits(final_model, Xva, tta=mode).argmax(1) == yva).float().mean():.4f}')
    pred = predict_logits(final_model, Xva, tta='full').argmax(1).cpu()
    print(pd.DataFrame(confusion_matrix(yva.cpu(), pred, labels=list(range(10))), index=CLASSES, columns=CLASSES))
else:
    print('No validation set (VAL_SIZE = 0) - skipping val accuracy.')

## 7. Predict & write `submission.csv`

In [ ]:
test_pred = predict_logits(final_model, Xte, tta='full' if TTA_FULL else 'flip').argmax(1).cpu().numpy()
sub = pd.DataFrame({'id': sample['id'], 'label': test_pred})
assert len(sub) == len(sample) and sub['label'].between(0, 9).all()
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub.head()); print('predictions per class:', sub['label'].value_counts().sort_index().tolist())
print('Saved /kaggle/working/submission.csv -> download it from the Output panel on the right, then Submit Prediction.')

## For the report (optional run)
Set `VAL_SIZE = 5000` and e.g. `TEACHER_EPOCHS = EPOCHS = 60`, then run. The validation summary prints the big teacher and the distilled student on the same held-out images — a clean "did distillation help?" comparison. Then go back to `VAL_SIZE = 0` and the default epochs for the final submission.